In [2]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utils import bootstrap_project_paths

project_root = bootstrap_project_paths()

from src.data import (
    load_inventory,
    load_shipment,
    load_orders,
    load_order_items,
    load_sales,
    load_promotions,
    load_products,
)

import matplotlib.pyplot as plt
import pandas as pd

DATA_ROOT = project_root / "data/datathon-2026-round-1"
inventory = load_inventory(DATA_ROOT)
shipment = load_shipment(DATA_ROOT)
orders = load_orders(DATA_ROOT)
order_items = load_order_items(DATA_ROOT)
sales = load_sales(DATA_ROOT)
promotions = load_promotions(DATA_ROOT)
products = load_products(DATA_ROOT)

d:\MyML\datathon2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: D:\MyML\datathon2026
Source path added: D:\MyML\datathon2026\src


In [ ]:
# list all unique order_status in orders
order_statuses = orders["order_status"].unique()
print("Unique order_status values in orders:")
print(order_statuses)
# count order_status na
order_status_na_count = orders["order_status"].isna().sum()
print(f"Number of orders with missing order_status: {order_status_na_count}")
# check if there are any order with order_id in orders but not in shipment
orders_with_shipment = orders[orders["order_id"].isin(shipment["order_id"])]
orders_without_shipment = orders[~orders["order_id"].isin(shipment["order_id"])]
print(f"Number of orders with shipment: {len(orders_with_shipment)}")

Unique order_status values in orders:
<StringArray>
['delivered', 'returned', 'shipped', 'cancelled', 'paid', 'created']
Length: 6, dtype: str
Number of orders with missing order_status: 0
Number of orders with shipment: 566067


Có 1708 ngày revenue lệch so với sales nếu tính tất cả trạng thái đơn hàng

Nếu loại trừ cancel và returned không tính vào revenue của ngày đó thì có hơn 2000 ngày lệch.

Chưa biết nguyên nhân gây lệch ở 1708 ngày.

In [6]:
# Compare daily revenue from delivered orders vs sales revenue.
order_items_rev = order_items.copy()
order_items_rev["product_revenue"] = (
    order_items_rev["unit_price"] * order_items_rev["quantity"] - order_items_rev["discount_amount"]
 )
order_rev_by_order = order_items_rev.groupby("order_id", as_index=False)["product_revenue"].sum()
orders_with_shipment = orders[orders["order_id"].isin(shipment["order_id"])].copy()
orders_delivered = orders_with_shipment[orders_with_shipment["order_status"] == "delivered"].copy()
orders_delivered_ids = orders_delivered[["order_id"]].drop_duplicates()
shipment_dates = shipment[["order_id", "delivery_date"]].copy()
shipment_dates["delivery_date"] = pd.to_datetime(shipment_dates["delivery_date"]).dt.date
shipment_dates = shipment_dates.dropna(subset=["delivery_date"])
shipment_dates = shipment_dates.groupby("order_id", as_index=False)["delivery_date"].max()
order_rev_by_order = order_rev_by_order.merge(orders_delivered_ids, on="order_id", how="inner")
order_rev_by_order = order_rev_by_order.merge(shipment_dates, on="order_id", how="inner")
order_rev_by_date = order_rev_by_order.groupby("delivery_date", as_index=False)["product_revenue"].sum()
order_rev_by_date.rename(columns={"delivery_date": "date", "product_revenue": "revenue_orders"}, inplace=True)
sales_rev_by_date = sales[["date", "Revenue"]].copy()
sales_rev_by_date["date"] = pd.to_datetime(sales_rev_by_date["date"]).dt.date
sales_rev_by_date.rename(columns={"Revenue": "revenue_sales"}, inplace=True)
revenue_compare = order_rev_by_date.merge(sales_rev_by_date, on="date", how="left")
revenue_compare["revenue_diff"] = revenue_compare["revenue_sales"] - revenue_compare["revenue_orders"]
revenue_compare["revenue_ratio"] = revenue_compare["revenue_sales"] / revenue_compare["revenue_orders"]
# sum revenue_diff
total_revenue_diff = revenue_compare["revenue_diff"].sum()
print(f"Total revenue difference (sales - orders): {total_revenue_diff:.2f}")
# print days where abs revenue_diff >= 0.1
diff_days = revenue_compare[revenue_compare["revenue_diff"].abs() >= 0.1]

print("Days with revenue difference:")
print(diff_days[["date", "revenue_orders", "revenue_sales", "revenue_diff", "revenue_ratio"]])  


Total revenue difference (sales - orders): 3914281781.58
Days with revenue difference:
            date  revenue_orders  revenue_sales  revenue_diff  revenue_ratio
0     2012-07-06       148168.15     3054029.42    2905861.27      20.611916
1     2012-07-07       520378.64     2667930.94    2147552.30       5.126903
2     2012-07-08      1075074.88     2360851.90    1285777.02       2.195988
3     2012-07-09      1027579.66     3548386.46    2520806.80       3.453150
4     2012-07-10      1300666.04     5234938.62    3934272.58       4.024814
...          ...             ...            ...           ...            ...
3826  2022-12-27      1000989.82     2100553.66    1099563.84       2.098477
3827  2022-12-28       877005.03     3448729.20    2571724.17       3.932394
3828  2022-12-29      1037049.40     3083944.33    2046894.93       2.973768
3829  2022-12-30      1153299.03     2884668.76    1731369.73       2.501232
3830  2022-12-31       941607.51     2383037.48    1441429.97     

In [11]:
# Diagnose revenue differences.
orders_only = revenue_compare[revenue_compare["revenue_sales"].isna()].copy()
sales_only = revenue_compare[revenue_compare["revenue_orders"].isna()].copy()
both = revenue_compare.dropna(subset=["revenue_orders", "revenue_sales"]).copy()

print("Days only in orders:", len(orders_only))
print("Days only in sales:", len(sales_only))
print("Days in both:", len(both))

print("Total revenue_orders:", both["revenue_orders"].sum())
print("Total revenue_sales:", both["revenue_sales"].sum())
print("Total diff on overlap:", (both["revenue_sales"] - both["revenue_orders"]).sum())

# Show top 10 absolute differences on overlapping dates.
top_diff = both.reindex(both["revenue_diff"].abs().sort_values(ascending=False).index)
top_diff.head(10)[["date", "revenue_orders", "revenue_sales", "revenue_diff", "revenue_ratio"]]

Days only in orders: 0
Days only in sales: 0
Days in both: 3833
Total revenue_orders: 15680952166.23
Total revenue_sales: 16430476585.529999
Total diff on overlap: 749524419.3


,date,revenue_orders,revenue_sales,revenue_diff,revenue_ratio
1821,2017-06-29,11952200.91,14575854.67,2623653.76,1.219512
1456,2016-06-29,10941875.87,13343751.12,2401875.25,1.219512
2186,2018-06-29,10236949.88,12484085.14,2247135.26,1.219512
1457,2016-06-30,9695774.58,11824115.33,2128340.75,1.219512
1730,2017-03-30,15414114.30,17516038.82,2101924.52,1.136364
2184,2018-06-27,9086540.11,11081146.52,1994606.41,1.219512
1820,2017-06-28,9075128.41,11067229.74,1992101.33,1.219512
1731,2017-03-31,14569862.68,16556662.03,1986799.35,1.136364
725,2014-06-29,8993137.01,10967240.25,1974103.24,1.219512
1091,2015-06-30,8982876.51,10954727.42,1971850.91,1.219512


In [ ]:
# List returned/canceled orders by day.
orders_rc = orders[orders["order_status"].isin(["returned", "canceled"])].copy()
orders_rc["order_date"] = pd.to_datetime(orders_rc["order_date"]).dt.date
orders_rc_by_day = (
    orders_rc.groupby(["order_date", "order_status"])
    .size()
    .reset_index(name="order_count")
    .sort_values(["order_date", "order_status"])
 )
orders_rc_by_day.head(20)

In [ ]:
grouped = order_items.groupby("product_id")["unit_price"].mean().reset_index()
merged = pd.merge(grouped, products[["product_id", "price"]], on="product_id", how="inner")
merged.head()


,product_id,unit_price,price
0,1,3912.790000,4945.500000
1,3,10203.173563,10831.377188
2,4,9303.285509,9610.756522
3,5,7673.590000,8946.000000
4,7,5013.258276,5306.997500


In [33]:
# promotions: promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
# order_items: order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
# orders: order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source

# Compute order-level revenue and profit (shipping fee is per order).
order_items["product_revenue"] = (
    order_items["unit_price"] * order_items["quantity"] - order_items["discount_amount"]
 )
order_revenue = order_items.groupby("order_id", as_index=False)["product_revenue"].sum()
shipping_fee_by_order = shipment.groupby("order_id", as_index=False)["shipping_fee"].sum()
orders_delivered = orders[orders["order_status"] == "delivered"].copy()
orders_dates = orders_delivered[["order_id", "date"]].copy()
orders_dates["date"] = pd.to_datetime(orders_dates["date"]).dt.date
order_revenue = order_revenue.merge(orders_dates, on="order_id", how="inner")
order_revenue = order_revenue.merge(shipping_fee_by_order, on="order_id", how="left")
order_revenue["shipping_fee"] = order_revenue["shipping_fee"].fillna(0)
order_revenue["profit"] = order_revenue["product_revenue"] - order_revenue["shipping_fee"]
order_revenue.head()

,order_id,product_revenue,date,shipping_fee,profit
0,1,7967.54,2012-07-04,1.37,7966.17
1,3,33660.99,2012-07-04,2.38,33658.61
2,4,53196.25,2012-07-04,2.49,53193.76
3,6,1597.84,2012-07-06,25.79,1572.05
4,7,9800.94,2012-07-06,1.31,9799.63


In [35]:
profit_by_date = order_revenue.groupby("date", as_index=False)["profit"].sum()
profit_by_date.rename(columns={"order_date": "date"}, inplace=True)
sales["date"] = pd.to_datetime(sales["date"]).dt.date
sales["profit_sales"] = sales["Revenue"] - sales["COGS"]
profit_by_date = profit_by_date.merge(sales[["date", "profit_sales"]], on="date", how="left")
profit_by_date

,date,profit,profit_sales
0,2012-07-04,4522520.67,1140556.75
1,2012-07-05,2358765.02,601193.22
2,2012-07-06,2174921.74,536396.58
3,2012-07-07,1901107.94,559684.32
4,2012-07-08,1782499.32,552229.11
...,...,...,...
3828,2022-12-27,1376844.61,-84318.58
3829,2022-12-28,2293060.92,-64891.80
3830,2022-12-29,1762662.50,-86842.77
3831,2022-12-30,1786534.54,-137623.39
